# 05-9. 웹 접근 로그 분석 종합 실습

## Goal

합성 Combined Log fixture를 스트리밍하고 인코딩·형식·의미 오류를 나눈다. 5분 시간창 특징과 NumPy 마스크로 교육용 조사 후보를 만든다.


## Setup

실제 로그를 사용하지 않고 문서용 IP 대역으로 만든 fixture를 사용한다. 풀이는 `notebooks/solutions/05-9-web-log-analysis-solution.ipynb`에서 자신의 구현 후 검증한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)

import numpy as np
import pandas as pd
from tempfile import TemporaryDirectory

LOG_PATHS = [
    FIXTURE_DIR / "web-access.log",
    FIXTURE_DIR / "web-access-invalid-utf8.log",
]
for path in LOG_PATHS:
    print(path.name, path.stat().st_size, "bytes")


## Steps

1. 바이트 줄에서 CRLF/LF 레코드 종료만 제거한다.
2. UTF-8 strict 디코딩 오류를 별도 건수로 나눈다.
3. 형식 파싱 후 `ipaddress.ip_address()`로 IPv4·IPv6를 검증한다.
4. URL raw path를 보존하되 파생된 정규화 경로로 집계한다.
5. 5분·IP별 특징표와 NumPy 조건 마스크를 만든다.
6. IP와 경로를 가명처리하고 전용 루트에 원자적으로 저장한다.


In [ ]:
TODO_DONE = False
MAX_LINE_BYTES = 16_384


def remove_record_separator(raw_line: bytes) -> bytes:
    # TODO: b"\\n" 문자 집합이 아니라 실제 CRLF/LF 접미사만 제거한다.
    raise NotImplementedError


def parse_combined_log(text: str) -> dict:
    # TODO: 형식 파싱과 의미 검증을 나눈다.
    raise NotImplementedError


def analyze_files(paths: list[Path]) -> tuple[list[dict], dict]:
    # TODO: 제한 readline, UTF-8 strict, 오류 유형별 건수로 fixture 전체를 처리한다.
    raise NotImplementedError


def build_features(records: list[dict]) -> pd.DataFrame:
    # TODO: 5분·IP별 특징을 반환한다.
    raise NotImplementedError


def classify_candidates(features: pd.DataFrame) -> pd.DataFrame:
    # TODO: np.divide, bool 마스크, np.select를 사용한다.
    raise NotImplementedError


def mask_features(features: pd.DataFrame, masking_key: bytes) -> pd.DataFrame:
    # TODO: HMAC 별칭을 만들고 원문 IP·경로·쿼리를 결과에서 제외한다.
    raise NotImplementedError


def publish_run(
    masked: pd.DataFrame,
    quality: dict,
    output_root: Path,
    run_id: str,
) -> Path:
    # TODO: 숨김 staging 디렉터리에 완성한 뒤 run 디렉터리로 한 번에 게시한다.
    raise NotImplementedError


## Checks

공개 경계 검증은 줄 종료·정상 IP·잘못된 IP·시간창 규칙을 확인한다. 탐지 후보는 침해 확정이 아니라 원본 맥락을 다시 볼 우선순위이다.


In [ ]:
if not TODO_DONE:
    print("TODO를 구현한 뒤 TODO_DONE을 True로 바꾸세요.")
else:
    assert remove_record_separator(b"record\r\n") == b"record"
    assert remove_record_separator(b"record\n") == b"record"
    assert remove_record_separator(b"login") == b"login"
    sample = (
        '203.0.113.10 - - [14/Aug/2026:10:30:00 +0900] '
        '"GET /login?token=secret HTTP/1.1" 200 443 "-" "TrainingBrowser/1.0"'
    )
    parsed = parse_combined_log(sample)
    assert parsed["ip"] == "203.0.113.10"
    assert parsed["normalized_path"] == "/login"

    records, quality = analyze_files(LOG_PATHS)
    assert quality == {
        "total_lines": 87,
        "parsed_lines": 82,
        "encoding_errors": 1,
        "format_errors": 1,
        "validation_errors": 3,
        "oversized_line_errors": 0,
        "known_bytes_rows": 81,
        "missing_bytes": 1,
    }
    features = classify_candidates(build_features(records))
    masked = mask_features(
        features,
        b"chapter05-learner-public-check-key",
    )
    assert "ip" not in masked.columns
    assert "source_alias" in masked.columns
    assert not masked.astype("string").apply(
        lambda column: column.str.contains("203.0.113.10", regex=False).any()
    ).any()

    with TemporaryDirectory(prefix="chapter05-learner-") as temporary_root:
        output_root = Path(temporary_root).resolve()
        run_dir = publish_run(masked, quality, output_root, "public-check")
        assert run_dir.resolve().parent == output_root
        assert {path.name for path in run_dir.iterdir()} == {
            "window-features.csv",
            "quality-report.json",
        }
        assert not list(output_root.glob(".staging-public-check-*"))
    print("공개 경계 검증 통과")


## Next Steps

자신의 구현을 풀이 검증용 Notebook의 품질 불변식과 비교한다. 실제 평가에서는 공개 저장소의 풀이본과 별개인 교수자용 비공개 테스트를 사용한다.
